# gpudb quick start — plain DuckDB SQL on the GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/singhpratech/duckdbgpumetaldbram/blob/main/examples/gpudb_quickstart.ipynb)

**gpudb** is a DuckDB extension that answers ordinary SQL on the GPU — an Apple
Silicon **Metal** backend and an NVIDIA **CUDA** backend behind one interface.
You write plain DuckDB SQL. Each statement goes to the device when that is
predicted to be faster and stays on DuckDB when it is not, and the answer is
the same either way.

This notebook, top to bottom:

1. `pip install duckdb-gpudb` — the wrapper, the `gpudb` shell command and, on
   Colab, the CUDA-enabled extension binary itself. No `INSTALL`, no build.
2. TPC-H SF1 through the connection — a 6-million-row `lineitem`.
3. The same plain SQL — a `GROUP BY`, a `WHERE` + `HAVING`, a join — run on
   **both** gpudb and a stock `duckdb` connection over the same file: rows
   compared row for row, times printed, and `con.last_rewrite()` saying where
   each statement ran and why.
4. The `gpudb` shell, which prints the same thing under every result.
5. The explicit `gpu_*` SQL functions, which any DuckDB client gets from the
   community registry without the wrapper.
6. An appendix: building the CUDA engine from source, for working on the
   engine itself.

On Colab's free **T4** runtime the wheel carries a CUDA build, so sections 1–4
use the GPU. **With no GPU attached nothing here fails** — every cell runs, on
DuckDB, with the same answers. No cell claims a backend it did not find: each
one prints what actually happened on the machine you are reading this on.

## 1. Plain SQL on the GPU in one `pip install`

On x86-64 Linux with glibc 2.34 or newer — which is what a Colab runtime is —
and on Apple Silicon, `pip install duckdb-gpudb` installs a platform wheel that
carries the extension binary inside the package. The Linux wheel is the
**CUDA-enabled** build and brings its own `libgomp`, so nothing else has to be
installed: with an NVIDIA driver present it uses the GPU, and without one it
falls back to the CPU backend cleanly.

In [ ]:
%pip install -q duckdb-gpudb

In [ ]:
import re
import duckdb
import gpudb

print("gpudb", gpudb.__version__, "· duckdb", duckdb.__version__)

probe = gpudb.connect()                    # in-memory, just to ask the extension what it is
info = probe.sql("SELECT gpu_build_info()").fetchall()[0][0]
note = probe.extension_note
probe.close()

print(info)

fields = dict(re.findall(r"(\w+)=('[^']*'|\S+)", info))
BACKEND = fields.get("runtime", "cpu")                     # cuda | metal | cpu
DEVICE = fields.get("device", "").strip("'")
NAMES = {"cuda": "NVIDIA CUDA", "metal": "Apple Silicon Metal"}

print()
if BACKEND in NAMES:
    print(f"GPU backend active: {NAMES[BACKEND]}" + (f" · {DEVICE}" if DEVICE else ""))
else:
    print("No GPU backend active — this extension reports runtime=cpu.")
    print("On Colab: Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all.")
    print("Either way, every cell below runs and every answer below is the same.")
print("extension:", note or "loaded and usable")

## 2. Data — TPC-H SF1

`tpch` is a core DuckDB extension, so `dbgen` is two statements away. SF1 is
about 6 million `lineitem` rows and takes well under a minute on a Colab VM.

It is written to a **file** rather than to memory for two reasons: the `gpudb`
shell in section 4 opens the same file, and a stock `duckdb` connection opens
it alongside so that every comparison below is over one copy of one dataset.

In [ ]:
import os
import time

DB = "tpch.duckdb"

if not os.path.exists(DB):
    t0 = time.perf_counter()
    build = gpudb.connect(DB)
    build.execute("INSTALL tpch")
    build.execute("LOAD tpch")
    build.execute("CALL dbgen(sf=1)")
    build.close()
    print(f"dbgen(sf=1) in {time.perf_counter() - t0:.1f} s")

# The wrapper: plain SQL in, GPU when it is predicted to win.
con = gpudb.connect(DB, read_only=True)

# Stock DuckDB over the same file, for the row-for-row and time comparison.
# DuckDB requires every connection to one file to share one configuration, which
# is why this repeats the setting gpudb.connect() uses to load its extension.
try:
    native = duckdb.connect(DB, read_only=True, config={"allow_unsigned_extensions": "true"})
except duckdb.Error:
    native = duckdb.connect(DB, read_only=True)

print("extension:", con.extension_note or "loaded and usable")
for t in ("lineitem", "orders"):
    n = con.execute(f"SELECT count(*) FROM {t}").fetchone()[0]
    print(f"{t:9s} {n:>10,} rows")

## 3. The same SQL, answered on the GPU

Three things are worth knowing before reading the output.

* **A statement is plain DuckDB SQL.** Nothing below is a gpudb function call.
* **The first ask usually runs on DuckDB.** Columns have to be on the device
  before a statement can use them, and in the default `background` residency
  the wrapper uploads them in short segments taken only while the connection is
  idle — so an upload never runs a long scan beside a query you are waiting on.
  The cell below asks each statement once, then waits for the uploads to land.
  (A script that knows its workload can pay the cost up front instead:
  `gpudb.connect(..., residency="eager")`.)
* **`con.last_rewrite()` is the receipt.** `rewritten` says whether the device
  answered; `reason` and `detail` say why not when it did not — `not_resident`
  (on its way), `threshold` (decided against, with the arithmetic) or `shape`
  (no kernel expresses it). A statement that stays on DuckDB is a decision, not
  a failure, and the answer is identical.

If this runtime has **no GPU**, the three comparisons below are DuckDB against
DuckDB: the wrapper carries thresholds for Metal and CUDA only, so every
statement reports `threshold: no thresholds for backend 'CPU'`, the times sit
on top of each other, and the rows still match exactly — which is the whole
promise.

In [ ]:
QUERIES = {
    "top-k GROUP BY":
        "SELECT l_partkey, sum(l_quantity) AS qty "
        "FROM lineitem GROUP BY l_partkey ORDER BY qty DESC LIMIT 5",
    "WHERE + HAVING":
        "SELECT l_partkey, sum(l_quantity) AS qty FROM lineitem WHERE l_quantity > 5 "
        "GROUP BY l_partkey HAVING sum(l_quantity) > 1200 ORDER BY l_partkey",
    "join + GROUP BY":
        "SELECT o_orderpriority, sum(l_extendedprice) AS revenue "
        "FROM lineitem JOIN orders ON l_orderkey = o_orderkey "
        "WHERE o_orderdate >= DATE '1994-01-01' AND o_orderdate < DATE '1995-01-01' "
        "GROUP BY 1 ORDER BY 1",
}


def where_it_ran(connection):
    """One line from con.last_rewrite(): where the statement ran, and why."""
    r = connection.last_rewrite()
    if r["rewritten"]:
        return f"GPU ({r['form']}: {r['detail']})"
    return f"DuckDB ({r['reason']}: {r['detail']})"


def timed(run, sql, reps=5):
    """Run it reps times; return the rows and the best wall time in ms."""
    times = []
    for _ in range(reps):
        t = time.perf_counter()
        rows = run(sql).fetchall()
        times.append((time.perf_counter() - t) * 1e3)
    return rows, min(times), times


def compare(name, reps=5, show=5):
    sql = QUERIES[name]
    print(sql + "\n")
    g_rows, g_ms, g_all = timed(con.execute, sql, reps)
    n_rows, n_ms, n_all = timed(native.execute, sql, reps)
    assert g_rows == n_rows, "gpudb and native DuckDB disagree on the rows"
    print(f"rows identical: True  ({len(g_rows)} rows)")
    for row in g_rows[:show]:
        print("   ", row)
    if len(g_rows) > show:
        print(f"    … {len(g_rows) - show} more")
    print(f"\nnative DuckDB  best {n_ms:6.1f} ms   {[round(x, 1) for x in n_all]}")
    print(f"gpudb          best {g_ms:6.1f} ms   {[round(x, 1) for x in g_all]}")
    print(f"               {n_ms / g_ms:.2f}x")
    print("where it ran:", where_it_ran(con))

In [ ]:
# Ask each statement once. This is what registers the columns it needs; the
# upload then runs in the background, and this first answer comes from DuckDB.
for name, sql in QUERIES.items():
    con.execute(sql).fetchall()
    print(f"{name:16s} first ask -> {where_it_ran(con)}")

# Wait for the background uploads. Politely: poll con.residents(), which reports
# missing / pending / uploading / ready / stale / failed per set. (With no GPU
# backend there is nothing to wait for — the wrapper has thresholds only for
# Metal and CUDA, so every statement stays on DuckDB.)
t0 = time.time()
if BACKEND in NAMES:
    while time.time() - t0 < 120:
        state = con.residents()
        if state and all(v in ("ready", "failed") for v in state.values()):
            break
        time.sleep(0.5)

print(f"\nresident sets after {time.time() - t0:.1f} s:")
for tag, st in con.residents().items():
    print(f"   {st:9s} {tag}")
if not con.residents():
    print("   (none — nothing was uploaded, and every statement stays on DuckDB)")

### 3a. A plain `GROUP BY` with `ORDER BY … LIMIT`

The top five part keys by quantity shipped, out of 200,000 keys over 6M rows.
On the device the group-by is a segmented reduce over a sort the GPU keeps, and
the `ORDER BY … LIMIT` runs there too, so only the five survivors come back.

In [ ]:
compare("top-k GROUP BY")

### 3b. `WHERE` + `HAVING`

The filter, the grouped sum and the `HAVING` are one pass on the device, and
only the groups that survive the `HAVING` are returned.

In [ ]:
compare("WHERE + HAVING")

### 3c. A join

`lineitem ⋈ orders` on the order key, filtered on the order date and grouped by
order priority. The join and the aggregate both run on the device against a
build side it keeps sorted.

In [ ]:
compare("join + GROUP BY")

## 4. The same thing from the `gpudb` shell

`pip install duckdb-gpudb` also puts a `gpudb` command on your `PATH`: a SQL
shell over the same wrapper, with one line under every result saying where the
statement ran and how long it took. That footer is on at a terminal and off in
scripted output, so `--timer` asks for it here.

The same statement runs twice. `--residency eager` uploads inside the statement
that first needs the column rather than in the background, so the first timing
includes the upload and the second is the hot one. With no GPU attached both
footers read `DuckDB`, with the reason spelled out, and the two results are the
same three rows.

In [ ]:
!gpudb tpch.duckdb --readonly --timer --residency eager \
    -c "SELECT l_returnflag, count(*) AS n FROM lineitem GROUP BY 1 ORDER BY 1;" \
    -c "SELECT l_returnflag, count(*) AS n FROM lineitem GROUP BY 1 ORDER BY 1;"

## 5. The explicit `gpu_*` functions, from any DuckDB client

Plain SQL reaches the GPU through the wrapper — `gpudb.connect()` or the
`gpudb` shell — because something has to see the statement before DuckDB plans
it. Everything gpudb can do is *also* reachable by name, from any DuckDB client
that can `LOAD` the extension: upload a column pair once, then ask for the
grouped result.

That route is `INSTALL gpudb FROM community;`. Read the build line the cell
prints: the registry's **Linux** binary is built without the CUDA toolchain, so
on Colab this section runs gpudb's CPU path — the same functions and the same
answers, just not the device. The GPU on Linux is the pip wheel in section 1.
(On an Apple Silicon Mac the same registry install is the full Metal build.)

In [ ]:
stock = duckdb.connect()                       # a stock DuckDB connection, no wrapper
stock.execute("INSTALL gpudb FROM community")
stock.execute("LOAD gpudb")

reg = stock.execute("SELECT gpu_build_info()").fetchall()[0][0]
print(reg)
reg_runtime = dict(re.findall(r"(\w+)=('[^']*'|\S+)", reg)).get("runtime", "cpu")
print("registry build runtime:", reg_runtime,
      "— the GPU" if reg_runtime in NAMES else "— the CPU path, same functions and same answers")

stock.execute("""CREATE TABLE sales AS
                 SELECT (range % 1000)::BIGINT AS store,
                        (range * 7 % 10007)::BIGINT AS amount
                 FROM range(10000000)""")
stock.execute("SELECT gpu_upload_pair('sales_by_store', store, amount) FROM sales")   # once

explicit = stock.execute(
    "SELECT * FROM gpu_groupby_sum_resident_topk('sales_by_store', 5, 'desc')").fetchall()
reference = stock.execute(
    "SELECT store, sum(amount) AS s, count(*) AS n "
    "FROM sales GROUP BY store ORDER BY s DESC, store LIMIT 5").fetchall()

print("\ngpu_groupby_sum_resident_topk:", explicit)
print("native GROUP BY             :", reference)
print("identical:", explicit == reference)
print(stock.execute("SELECT gpu_last_stats()").fetchall()[0][0])

## 6. Appendix — building the CUDA engine from source

**You do not need this to use the GPU on Colab**: the wheel in section 1 already
carries a CUDA build. This section is for working on the engine — compiling the
kernels, running the unit tests that check every backend against the CPU
reference, and running the operator-level benchmarks. It takes a few minutes on
Colab's 2-core VM.

Each cell below is a no-op on a runtime with no NVIDIA GPU, so that no CPU
number is ever printed under a CUDA heading.

In [ ]:
# Does this runtime actually have an NVIDIA GPU and a CUDA compiler?
import shutil
import subprocess

os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
CUDA_OK = bool(shutil.which("nvidia-smi")) and bool(shutil.which("nvcc"))

if CUDA_OK:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
    print(subprocess.run(["nvcc", "--version"], capture_output=True,
                         text=True).stdout.strip().splitlines()[-1])
else:
    print("No nvidia-smi / nvcc on this runtime — the rest of the appendix is skipped.")
    print("On Colab: Runtime -> Change runtime type -> T4 GPU to build the CUDA backend.")

In [ ]:
# GPUDB_REQUIRE_CUDA=1 makes configure FAIL if CMake cannot find nvcc, so there is
# no silent CPU-only build. CUDAARCHS=75 compiles only the T4's sm_75, which is much
# faster on a 2-core VM. Look for 'CUDA enabled (compiler: ...)' in the output.
if CUDA_OK:
    !git clone --depth 1 https://github.com/singhpratech/duckdbgpumetaldbram.git
    !cd duckdbgpumetaldbram && CUDAARCHS=75 GPUDB_REQUIRE_CUDA=1 ./scripts/build.sh 2>&1 | tee build.log | grep -E 'CUDA enabled|CUDA toolkit not found|error|Error|==>'
    !grep -q 'CUDA enabled' duckdbgpumetaldbram/build.log && echo 'OK: CUDA backend compiled in' || (echo 'FAIL: CUDA backend NOT compiled — see build.log'; exit 1)
else:
    print("skipped — no CUDA toolchain on this runtime")

In [ ]:
# Unit tests: every compiled backend against the CPU reference.
# Expect 'available backends: CPU CUDA' and a '--- testing backend: CUDA ---' section.
if CUDA_OK:
    !cd duckdbgpumetaldbram && ./build-linux/test/test_gpudb | tee test.log
    !grep -q 'available backends:.*CUDA' duckdbgpumetaldbram/test.log && echo 'OK: tests ran on CUDA' || (echo 'FAIL: CUDA backend not available at runtime'; exit 1)
else:
    print("skipped — no CUDA toolchain on this runtime")

In [ ]:
# Resident-column aggregate benchmark: CPU vs CUDA on this GPU.
if CUDA_OK:
    !cd duckdbgpumetaldbram && ./build-linux/bin/gpudb-bench
else:
    print("skipped — no CUDA toolchain on this runtime")

In [ ]:
# GROUP BY at high cardinality — 50M rows, 10M groups.
if CUDA_OK:
    !cd duckdbgpumetaldbram && ./build-linux/bin/gpudb-groupby-bench --rows 50000000 --groups 10000000
else:
    print("skipped — no CUDA toolchain on this runtime")

## Where to go next

* **README** — what it is, how it decides, and the numbers with their
  conditions: <https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/README.md>
* **The Python way** — every `gpudb.connect()` option, `last_rewrite()`,
  residency, memory, threads: <https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/docs/USING_PYTHON.md>
* **The `gpudb` shell** — the footer, `.gpu`, `.residents`, `.memory`, the
  flags: <https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/docs/USING_THE_SHELL.md>
* **Installing** — both routes, the lookup order, supported versions,
  troubleshooting: <https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/docs/INSTALL.md>
* **Benchmarks**, append-only and with reproduction steps:
  <https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/BENCHMARK.md>
* **Known limitations**: <https://github.com/singhpratech/duckdbgpumetaldbram/blob/main/KNOWN_ISSUES.md>
* The repository itself: <https://github.com/singhpratech/duckdbgpumetaldbram>